# CatBoost Classifier
**DSC 148 Final Project**

CatBoost is Yandex's gradient-boosted tree implementation with two distinctive features relevant to this dataset: (1) native ordered target statistics for categorical features (avoids target leakage that plain label encoding can introduce), and (2) symmetric (oblivious) tree structure that provides strong regularization. Here the label-encoded categoricals are passed as `cat_features` so CatBoost can apply its own internal encoding rather than treating integer codes as continuous.

In [6]:
#!pip install pandas numpy matplotlib scikit-learn catboost -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, os
from catboost import CatBoostClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
SEED = 42
print('CatBoost loaded.')

CatBoost loaded.


## 1. Data Loading and Preprocessing

In [7]:
def reduce_mem(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    return df

train_trans = reduce_mem(pd.read_csv('data/train_transaction.csv'))
train_id    = reduce_mem(pd.read_csv('data/train_identity.csv'))
df = train_trans.merge(train_id, on='TransactionID', how='left')

START_DATE = pd.Timestamp('2017-11-30')
dt = START_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
df['tx_hour']  = dt.dt.hour.astype('int8')
df['tx_dow']   = dt.dt.dayofweek.astype('int8')
df['tx_month'] = dt.dt.month.astype('int8')
df['card_age'] = (df['TransactionDT'] // 86400 - df['D1']).astype('float32')
print(f'Merged shape: {df.shape}')

Merged shape: (590540, 438)


In [ ]:
df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)
n       = len(df_sorted)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_df = df_sorted.iloc[:n_train].copy()
val_df   = df_sorted.iloc[n_train:n_train+n_val].copy()
test_df  = df_sorted.iloc[n_train+n_val:].copy()

y_train = train_df['isFraud'].values
y_val   = val_df['isFraud'].values
y_test  = test_df['isFraud'].values

print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}')

# Label-encode for consistent column dtypes; CatBoost will re-encode cat_features internally
CAT_COLS = [
    'ProductCD','card4','card6','P_emaildomain','R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'DeviceType','DeviceInfo',
    'id_12','id_15','id_16','id_23','id_27','id_28',
    'id_29','id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38'
]
CAT_COLS = [c for c in CAT_COLS if c in train_df.columns]
for col in CAT_COLS:
    for part in [train_df, val_df, test_df]:
        part[col] = part[col].fillna('unknown').astype(str)
    cats = train_df[col].unique().tolist()
    for part in [train_df, val_df, test_df]:
        part[col] = pd.Categorical(part[col], categories=cats).codes
print(f'Label-encoded {len(CAT_COLS)} columns.')

In [ ]:
for part in [train_df, val_df, test_df]:
    part['uid'] = (part['card1'].astype(str) + '_' +
                   part['addr1'].fillna(-1).astype(int).astype(str) + '_' +
                   part['card_age'].fillna(-1).round(0).astype(int).astype(str))

c_cols    = [f'C{i}' for i in range(1, 15)]
uid_stats = (train_df.groupby('uid')
               .agg(uid_tx_count=('TransactionID','count'),
                    uid_amt_mean=('TransactionAmt','mean'),
                    uid_amt_std =('TransactionAmt','std'),
                    **{f'uid_{c}_mean': (c,'mean') for c in c_cols})
               .reset_index())
fallback = {
    'uid_tx_count': 1,
    'uid_amt_mean': float(train_df['TransactionAmt'].mean()),
    'uid_amt_std' : float(train_df['TransactionAmt'].std()),
    **{f'uid_{c}_mean': float(train_df[c].mean()) for c in c_cols},
}

def attach_uid(df_part, stats, fb):
    out = df_part.merge(stats, on='uid', how='left')
    for col, val in fb.items():
        out[col] = out[col].fillna(val).astype('float32')
    return out.drop(columns=['uid'])

train_df = attach_uid(train_df, uid_stats, fallback)
val_df   = attach_uid(val_df,   uid_stats, fallback)
test_df  = attach_uid(test_df,  uid_stats, fallback)

DROP = ['TransactionID','isFraud','TransactionDT']
v_cols = [c for c in train_df.columns if c.startswith('V')]
v_keep = [c for c in v_cols if train_df[c].isnull().mean() < 0.50]
v_drop = [c for c in v_cols if c not in v_keep]
FULL_FEATURES = [c for c in train_df.columns if c not in DROP and c not in v_drop]

# Column indices of categoricals within FULL_FEATURES (for CatBoost)
cat_feature_names = [c for c in CAT_COLS if c in FULL_FEATURES]
print(f'Features: {len(FULL_FEATURES)}  |  Categoricals passed to CatBoost: {len(cat_feature_names)}')

def _full(df_part):
    X = df_part[FULL_FEATURES].copy()
    X['TransactionAmt'] = np.log1p(X['TransactionAmt'])
    return X

X_train_f = _full(train_df)
X_val_f   = _full(val_df)
X_test_f  = _full(test_df)

## 2. CatBoost

`cat_features` tells CatBoost which columns to treat as categoricals -- it applies ordered target statistics internally, which can outperform plain label encoding. `scale_pos_weight` handles imbalance. Early stopping on val AUC.

In [ ]:
neg, pos = np.bincount(y_train)
spw = neg / pos
print(f'scale_pos_weight = {spw:.1f}')

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.02,
    depth=8,
    scale_pos_weight=spw,
    subsample=0.8,
    rsm=0.7,
    min_data_in_leaf=50,
    l2_leaf_reg=1.0,
    random_seed=SEED,
    eval_metric='AUC',
    early_stopping_rounds=50,
    verbose=100,
    allow_writing_files=False
)
cat_model.fit(
    X_train_f, y_train,
    eval_set=(X_val_f, y_val),
    cat_features=cat_feature_names
)

y_prob_cat_val = cat_model.predict_proba(X_val_f)[:, 1]
y_prob_cat     = cat_model.predict_proba(X_test_f)[:, 1]

# Threshold pre-tuned on the validation set via precision_recall_curve.
# Hardcoded to avoid re-running the search on every execution.
# To re-derive: uncomment the block below and run once.
#
# from sklearn.metrics import precision_recall_curve
# prec_c, rec_c, thr_c = precision_recall_curve(y_val, y_prob_cat_val)
# f1_c = 2 * prec_c[:-1] * rec_c[:-1] / (prec_c[:-1] + rec_c[:-1] + 1e-9)
# best_cat = float(thr_c[np.argmax(f1_c)])
# print(f'Threshold (val): {best_cat:.4f}')

best_cat = 0.7722  # optimal val-F1 threshold found during initial run
y_pred_cat = (y_prob_cat >= best_cat).astype(int)
print(f'Threshold (val): {best_cat:.4f}')

print('='*35)
print('CatBoost -- Test Set')
print('='*35)
print(classification_report(y_test, y_pred_cat, target_names=['Legitimate','Fraud']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_prob_cat):.4f}')

## 3. Results

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_cat)
auc = roc_auc_score(y_test, y_prob_cat)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr, tpr, color='#D65F5F', label=f'CatBoost (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--',linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('CatBoost ROC Curve')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('results', exist_ok=True)

metrics = {
    'model'    : 'CatBoost',
    'accuracy' : float(accuracy_score(y_test, y_pred_cat)),
    'precision': float(precision_score(y_test, y_pred_cat, zero_division=0)),
    'recall'   : float(recall_score(y_test, y_pred_cat, zero_division=0)),
    'f1'       : float(f1_score(y_test, y_pred_cat, zero_division=0)),
    'auc'      : float(roc_auc_score(y_test, y_prob_cat)),
    'threshold': best_cat,
}
with open('results/catboost_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
np.save('results/catboost_probs.npy',     y_prob_cat)
np.save('results/catboost_val_probs.npy', y_prob_cat_val)
print('Saved results/catboost_metrics.json, catboost_probs.npy, catboost_val_probs.npy')
print(json.dumps(metrics, indent=2))

## 4. Comparison to Baselines

In [ ]:
# Load metrics from previously run notebooks
BASELINE_FILES = {
    'Logistic Regression': 'results/lr_metrics.json',
    'Naive Bayes'        : 'results/nb_metrics.json',
}

def load_metrics(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None

def load_probs(path):
    try:
        return np.load(path)
    except FileNotFoundError:
        return None

rows = {}
prob_files = {
    'Logistic Regression': 'results/lr_probs.npy',
    'Naive Bayes'        : 'results/nb_probs.npy',
}
for name, path in BASELINE_FILES.items():
    m = load_metrics(path)
    if m:
        rows[name] = {
            'Precision (Fraud)': m['precision'],
            'Recall (Fraud)'   : m['recall'],
            'F1 (Fraud)'       : m['f1'],
            'AUC-ROC'          : m['auc'],
        }

rows['CatBoost'] = {
    'Precision (Fraud)': round(float(precision_score(y_test, y_pred_cat, zero_division=0)), 4),
    'Recall (Fraud)'   : round(float(recall_score(y_test, y_pred_cat, zero_division=0)), 4),
    'F1 (Fraud)'       : round(float(f1_score(y_test, y_pred_cat, zero_division=0)), 4),
    'AUC-ROC'          : round(float(roc_auc_score(y_test, y_prob_cat)), 4),
}

cmp_df = pd.DataFrame(rows).T
print('Comparison Table (test set):')
print(cmp_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart
cmp_df[['F1 (Fraud)','AUC-ROC']].plot(
    kind='bar', ax=axes[0], color=['#D65F5F','#4878CF'], edgecolor='white', width=0.7)
axes[0].set_title('F1 and AUC-ROC: 'CatBoost' vs. Baselines')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15, ha='right')
axes[0].set_ylim(0, 1.05)
axes[0].legend()

# ROC curves
COLORS_CMP = ['#F0A202', '#D65F5F', '#808080', '#4878CF']
baseline_probs = {k: load_probs(v) for k, v in prob_files.items()}
for i, (name, yp) in enumerate(baseline_probs.items()):
    if yp is not None:
        fpr, tpr, _ = roc_curve(y_test, yp)
        axes[1].plot(fpr, tpr, color=COLORS_CMP[i], linewidth=1.5,
                     label=f"{name} ({rows[name]['AUC-ROC']:.3f})")
fpr, tpr, _ = roc_curve(y_test, y_prob_cat)
axes[1].plot(fpr, tpr, color='#4878CF', linewidth=2.5,
             label=f"'CatBoost' ({rows['CatBoost']['AUC-ROC']:.3f})")
axes[1].plot([0,1],[0,1],'k--',linewidth=0.8)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()